# Week 3 Day 3 - The context layer and memory

## Running UnderCover with its context layer on AWS

Today the claim agent moves onto the Gateway's tools. We deploy the new app, put UnderCover's context layer on Lambda as its own MCP server, create the Gateway again with both as targets, give the agent a Memory of its own, and watch a returning customer's second claim recall the first.

The agent now works from three sources of data. The context layer holds the business's data from other systems, which the agent reads through the Gateway. The Memory holds the agent's own notes on each customer. The database on Supabase is the system of record.

## 6 parts:

PART 1: The new app on Lambda

PART 2: The context layer on Lambda

PART 3: The Gateway with both targets

PART 4: The Memory

PART 5: A returning customer

PART 6: Teardown

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">You need the AWS account and the <code>undercover</code> IAM user from Day 1, and today's build of the product, yours or <code>week3day3-end</code>, with <code>UNDERCOVER_API_KEY</code> in your <code>.env</code> file. The app on Lambda from Day 2 is updated in place, and if you deleted it, Part 1 creates it again.<br/><br/>The app and the context layer are inside AWS's free allowance for Lambda. The Gateway, the Policy, the secret that holds the API key and the Memory cost a fraction of a cent while they exist, and Part 6 deletes them. The two claims in Part 5 cost a few cents on OpenRouter. Part 5 also reseeds your database, which starts the queue, the scoreboard and the configuration's versions afresh.<br/><br/>If you're on the free route, <a href="week3day3free.ipynb">week3day3free.ipynb</a> runs the context layer on your laptop beside the agent instead. The Gateway and the Memory need AWS, so for Parts 3 to 5 you can watch me.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Watching your AWS spend</h2>
            <span style="color:#118AB2;">Before you start, open a terminal in the <code>undercover</code> folder and run <code>uv run monitor.py</code>. It opens a page in your browser with what your AWS account has spent today and over the last 30 days, with and without your credits. AWS updates the figures up to three times a day.<br/><br/>For the official source of truth, please review Billing and Cost Management in your AWS console. Always have budgets with billing alerts, like the one from Day 1.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import asyncio
import json
import os
import re
import shutil
import subprocess
import time
from pathlib import Path

import boto3
import httpx
import requests
import yaml
from botocore.auth import SigV4Auth as BotocoreSigV4Auth
from botocore.awsrequest import AWSRequest
from dotenv import load_dotenv, set_key, unset_key
from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError

## Connect to AWS

boto3 uses the access key that `aws configure` saved on Day 1, in us-east-1.

In [ ]:
load_dotenv(override=True)

session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

In [ ]:
lambda_client = session.client("lambda")
iam = session.client("iam")
logs = session.client("logs")
agentcore = session.client("bedrock-agentcore-control")
memory_records = session.client("bedrock-agentcore")

Everything we create is named `undercover...` and tagged `project=undercover`, so the teardown can find it by name.

In [ ]:
APP_FUNCTION = "undercover-app"
APP_ROLE_NAME = "undercover-app-role"
CONTEXT_FUNCTION = "undercover-context"
CONTEXT_ROLE_NAME = "undercover-context-role"
KEY_PROVIDER_NAME = "undercover-app-key"
GATEWAY_ROLE_NAME = "undercover-gateway-role"
GATEWAY_NAME = "undercover-gateway"
APP_TARGET = "undercover-app"
CONTEXT_TARGET = "undercover-context"
ENGINE_NAME = "undercover_policy_engine"
MEMORY_NAME = "undercover_memory"
PROJECT_TAG = {"project": "undercover"}
ENV_FILE = Path("../.env")

In [ ]:
for name in ["OPENROUTER_API_KEY", "DATABASE_URL", "VITE_SUPABASE_URL", "VITE_SUPABASE_PUBLISHABLE_KEY", "UNDERCOVER_API_KEY"]:
    print(f"{name}: {'found' if os.getenv(name) else 'missing, please add it to .env'}")

API_KEY = os.environ["UNDERCOVER_API_KEY"]

## Part 1: The new app on Lambda

Today's build moved the document search and the section reading out of the app and into the context layer, and the decision record now checks itself against the claims manual. So the app on Lambda needs the new code. This is Day 2's deployment again, and it updates the function in place, which keeps its address.

### Building the dashboard and the package

In [ ]:
!cd .. && npm run build

In [ ]:
BUILD = Path("../build")
APP_PACKAGE = BUILD / "lambda"
APP_ZIP = BUILD / "undercover-app.zip"

shutil.rmtree(APP_PACKAGE, ignore_errors=True)
APP_PACKAGE.mkdir(parents=True)

!uv export --only-group server --no-hashes --no-emit-project --quiet --output-file {BUILD / "requirements.txt"}
!uv pip install --quiet --target {APP_PACKAGE} --python-platform x86_64-manylinux2014 --python-version 3.12 --only-binary :all: -r {BUILD / "requirements.txt"}

for folder in ["backend", "data"]:
    shutil.copytree(Path("..") / folder, APP_PACKAGE / folder, ignore=shutil.ignore_patterns("tests", "__pycache__"))
shutil.copytree(Path("../dashboard/dist"), APP_PACKAGE / "dashboard" / "dist")

shutil.make_archive(str(APP_ZIP.with_suffix("")), "zip", APP_PACKAGE)
print(f"{APP_ZIP.name}: {APP_ZIP.stat().st_size / 1_000_000:.1f} MB (Lambda takes up to 50 MB)")

### Three helpers for Lambda

We deploy two functions today, the app and the context layer, in the same way as Day 2: a log group with a retention period, a role that can only write to it, and the function, created the first time and updated in place after that.

In [ ]:
def make_log_group(name):
    """Create a log group that keeps logs for 7 days."""
    try:
        logs.create_log_group(logGroupName=name)
        print(f"Created log group {name}")
    except logs.exceptions.ResourceAlreadyExistsException:
        print(f"Log group {name} already exists")
    logs.put_retention_policy(logGroupName=name, retentionInDays=7)


def make_function_role(role_name, log_group, description):
    """Create a role that lets Lambda run a function and write to its log group, and nothing else."""
    lambda_trust = {
        "Version": "2012-10-17",
        "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}],
    }
    try:
        role = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(lambda_trust),
            Description=description,
            Tags=[{"Key": key, "Value": value} for key, value in PROJECT_TAG.items()],
        )["Role"]
        print(f"Created role {role_name}")
    except iam.exceptions.EntityAlreadyExistsException:
        role = iam.get_role(RoleName=role_name)["Role"]
        print(f"Role {role_name} already exists")
    logging_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["logs:CreateLogStream", "logs:PutLogEvents"],
                "Resource": f"arn:aws:logs:{region}:{account}:log-group:{log_group}:*",
            }
        ],
    }
    iam.put_role_policy(RoleName=role_name, PolicyName=f"{role_name}-logging", PolicyDocument=json.dumps(logging_policy))
    return role

In [ ]:
def deploy_function(name, role, zip_path, settings):
    """Create the function, waiting for a new role to reach Lambda, or update it in place."""
    zip_bytes = zip_path.read_bytes()
    try:
        lambda_client.get_function(FunctionName=name)
        lambda_client.update_function_code(FunctionName=name, ZipFile=zip_bytes)
        lambda_client.get_waiter("function_updated_v2").wait(FunctionName=name)
        lambda_client.update_function_configuration(FunctionName=name, Role=role["Arn"], **settings)
        lambda_client.get_waiter("function_updated_v2").wait(FunctionName=name)
        print(f"Updated function {name}")
    except lambda_client.exceptions.ResourceNotFoundException:
        for attempt in range(10):
            try:
                lambda_client.create_function(FunctionName=name, Role=role["Arn"], Code={"ZipFile": zip_bytes}, Tags=PROJECT_TAG, **settings)
                break
            except lambda_client.exceptions.InvalidParameterValueException:
                print("Waiting for the new role to reach Lambda")
                time.sleep(3)
        lambda_client.get_waiter("function_active_v2").wait(FunctionName=name)
        print(f"Created function {name}")
    return lambda_client.get_function(FunctionName=name)["Configuration"]["FunctionArn"]

### The app's function

The same settings as Day 2: the app reads the same Supabase database as your laptop, through the transaction pooler on port 6543, with a connection for each request.

In [ ]:
APP_LOG_GROUP = f"/aws/lambda/{APP_FUNCTION}"
make_log_group(APP_LOG_GROUP)
app_role = make_function_role(APP_ROLE_NAME, APP_LOG_GROUP, "The role the UnderCover app runs as on Lambda")

app_settings = {
    "Runtime": "python3.12",
    "Handler": "backend.main.handler",
    "Timeout": 30,
    "MemorySize": 1024,
    "Environment": {
        "Variables": {
            "DATABASE_URL": os.environ["DATABASE_URL"].replace(":5432/", ":6543/"),
            "VITE_SUPABASE_URL": os.environ["VITE_SUPABASE_URL"],
            "UNDERCOVER_API_KEY": API_KEY,
        }
    },
}

app_arn = deploy_function(APP_FUNCTION, app_role, APP_ZIP, app_settings)

The app keeps its public Function URL from Day 2. If it's new, it needs the two permissions that let anyone reach it, which is why the agent's routes need the API key and the handlers sign in.

In [ ]:
try:
    app_url_config = lambda_client.create_function_url_config(FunctionName=APP_FUNCTION, AuthType="NONE")
except lambda_client.exceptions.ResourceConflictException:
    app_url_config = lambda_client.get_function_url_config(FunctionName=APP_FUNCTION)

public_access = [
    {"StatementId": "public-url", "Action": "lambda:InvokeFunctionUrl", "FunctionUrlAuthType": "NONE"},
    {"StatementId": "public-invoke", "Action": "lambda:InvokeFunction", "InvokedViaFunctionUrl": True},
]

for statement in public_access:
    try:
        lambda_client.add_permission(FunctionName=APP_FUNCTION, Principal="*", **statement)
    except lambda_client.exceptions.ResourceConflictException:
        pass

app_url = app_url_config["FunctionUrl"].rstrip("/")
print(app_url)

### The agent's own API document

The agent's API document now has six operations, since searching and reading the documents moved to the context layer. Each becomes a tool on the Gateway.

In [ ]:
print(requests.get(f"{app_url}/api/health", timeout=60).json())

agent_api = requests.get(f"{app_url}/api/agent/openapi.json", headers={"X-API-Key": API_KEY}, timeout=30).json()
agent_api["servers"] = [{"url": app_url}]

for path, operations in agent_api["paths"].items():
    for method, operation in operations.items():
        print(f"{method.upper():<6}{operation['operationId']:<20}{path}")

## Part 2: The context layer on Lambda

### Seven files from across the business

The context layer gives the agent the business's own knowledge: what the data means, how the work is done, and where each piece came from. In UnderCover it's the seven files in `context/`, each standing in for another system at the firm or an outside provider, and each naming its source system, its owner and when it was last reviewed.

In [ ]:
def source_of(path):
    """A context file's source: the top of a YAML file, or a document's front matter."""
    text = path.read_text(encoding="utf-8")
    if path.suffix == ".md":
        text = text.split("---")[1]
    return yaml.safe_load(text)["source"]


CONTEXT_FILES = [
    "data_dictionary.yaml",
    "glossary.yaml",
    "plans.yaml",
    "claim_rules.yaml",
    "reference_data.yaml",
    "policy_wording.md",
    "claims_manual.md",
]

print(f"{'File':<22}{'Source system':<38}{'Owner':<28}Reviewed")
for name in CONTEXT_FILES:
    source = source_of(Path("../context") / name)
    print(f"{name:<22}{source['system']:<38}{source['owner']:<28}{source['reviewed']}")

### Preparing the documents

The agent doesn't read the policy wording or the claims manual whole. The context layer splits each document at its numbered headings into sections, each tagged with its document, its number and its title, so the agent can search them and read just the section it needs. This is the same split, on the claims manual.

In [ ]:
manual = Path("../context/claims_manual.md").read_text(encoding="utf-8")

for number, title in re.findall(r"^## (\d+)\. (.+)$", manual, flags=re.MULTILINE):
    print(f"claims_manual  section {number:>2}  {title}")

### The package

The context layer has its own dependency group in `pyproject.toml`, `context`, with only what its server needs. The package is the `context` folder, with the server's code and the seven files, and those packages for Linux.

In [ ]:
CONTEXT_PACKAGE = BUILD / "context"
CONTEXT_ZIP = BUILD / "undercover-context.zip"

shutil.rmtree(CONTEXT_PACKAGE, ignore_errors=True)
CONTEXT_PACKAGE.mkdir(parents=True)

!uv export --only-group context --no-hashes --no-emit-project --quiet --output-file {BUILD / "context-requirements.txt"}
!uv pip install --quiet --target {CONTEXT_PACKAGE} --python-platform x86_64-manylinux2014 --python-version 3.12 --only-binary :all: -r {BUILD / "context-requirements.txt"}

shutil.copytree(Path("../context"), CONTEXT_PACKAGE / "context", ignore=shutil.ignore_patterns("tests", "__pycache__"))

shutil.make_archive(str(CONTEXT_ZIP.with_suffix("")), "zip", CONTEXT_PACKAGE)
print(f"{CONTEXT_ZIP.name}: {CONTEXT_ZIP.stat().st_size / 1_000_000:.1f} MB")

### The function

The context layer runs on Lambda like the app, with no environment variables: everything it serves is in its package.

In [ ]:
CONTEXT_LOG_GROUP = f"/aws/lambda/{CONTEXT_FUNCTION}"
make_log_group(CONTEXT_LOG_GROUP)
context_role = make_function_role(CONTEXT_ROLE_NAME, CONTEXT_LOG_GROUP, "The role UnderCover's context layer runs as on Lambda")

context_settings = {"Runtime": "python3.12", "Handler": "context.server.handler", "Timeout": 30, "MemorySize": 512}

context_arn = deploy_function(CONTEXT_FUNCTION, context_role, CONTEXT_ZIP, context_settings)

### A Function URL with IAM auth

Unlike the app's, the context layer's Function URL uses `AWS_IAM` auth: a request has to be signed with AWS credentials allowed to invoke the function. The Gateway will be allowed, and so is your IAM user. Anyone without that permission is refused.

In [ ]:
try:
    context_url = lambda_client.create_function_url_config(FunctionName=CONTEXT_FUNCTION, AuthType="AWS_IAM")["FunctionUrl"]
except lambda_client.exceptions.ResourceConflictException:
    context_url = lambda_client.get_function_url_config(FunctionName=CONTEXT_FUNCTION)["FunctionUrl"]

context_url = context_url.rstrip("/")
print(context_url)

### Signing requests

The same class as Day 2. It signs each HTTP request with AWS Signature Version 4, as boto3 signs its own calls, for whichever AWS service the request is for: `lambda` for a Function URL, `bedrock-agentcore` for the Gateway.

In [ ]:
class SigV4Auth(httpx.Auth):
    """Sign each request with AWS Signature Version 4, as boto3 does for its own calls."""

    requires_request_body = True

    def __init__(self, session, service="bedrock-agentcore"):
        self.credentials = session.get_credentials()
        self.region = session.region_name
        self.service = service

    def auth_flow(self, request):
        headers = {"Host": request.url.netloc.decode(), "Content-Type": request.headers.get("content-type", "")}
        aws_request = AWSRequest(method=request.method, url=str(request.url), data=request.content, headers=headers)
        BotocoreSigV4Auth(self.credentials.get_frozen_credentials(), self.service, self.region).add_auth(aws_request)
        for name in ("Authorization", "X-Amz-Date", "X-Amz-Security-Token"):
            if name in aws_request.headers:
                request.headers[name] = aws_request.headers[name]
        yield request

### Trying it

An MCP request is a JSON message posted to `/mcp`. Without a signature, the Function URL refuses it. Signed with your IAM user, the context layer lists its tools. The first request takes a few seconds while Lambda starts the function.

In [ ]:
list_tools = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
mcp_headers = {"Accept": "application/json, text/event-stream"}

unsigned = httpx.post(f"{context_url}/mcp", json=list_tools, headers=mcp_headers, timeout=60)
print(f"Without a signature: {unsigned.status_code}")

signed = httpx.post(f"{context_url}/mcp", json=list_tools, headers=mcp_headers, auth=SigV4Auth(session, "lambda"), timeout=60)
print(f"Signed: {signed.status_code}\n")

for tool in signed.json()["result"]["tools"]:
    print(f"{tool['name']:<24}{tool['description']}")

## Part 3: The Gateway with both targets

Day 2's teardown deleted the Gateway, so we create it again, this time with two targets: the app, as on Day 2, and the context layer.

### The API key in AgentCore Identity

As on Day 2, the Gateway sends the app's API key with every call to the app, from an API key credential provider.

In [ ]:
def find_key_providers():
    """The API key credential providers named undercover..."""
    pages = agentcore.get_paginator("list_api_key_credential_providers").paginate()
    return [p["name"] for page in pages for p in page["credentialProviders"] if p["name"].startswith("undercover")]


if KEY_PROVIDER_NAME in find_key_providers():
    agentcore.update_api_key_credential_provider(name=KEY_PROVIDER_NAME, apiKey=API_KEY)
    print(f"Credential provider {KEY_PROVIDER_NAME} already exists, key updated")
else:
    agentcore.create_api_key_credential_provider(name=KEY_PROVIDER_NAME, apiKey=API_KEY, tags=PROJECT_TAG)
    print(f"Created credential provider {KEY_PROVIDER_NAME}")

provider = agentcore.get_api_key_credential_provider(name=KEY_PROVIDER_NAME)
provider_arn = provider["credentialProviderArn"]
secret_arn = provider["apiKeySecretArn"]["secretArn"]

### The Gateway's role

Day 2's role, with one more permission: to invoke the context layer's function. The Gateway signs its calls to the context layer with this role, so the context layer needs no key of its own.

In [ ]:
gateway_trust = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account}:gateway/{GATEWAY_NAME}-*"},
            },
        }
    ],
}

try:
    gateway_role = iam.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(gateway_trust),
        Description="The role the UnderCover gateway runs as",
        Tags=[{"Key": key, "Value": value} for key, value in PROJECT_TAG.items()],
    )["Role"]
    print(f"Created role {GATEWAY_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    gateway_role = iam.get_role(RoleName=GATEWAY_ROLE_NAME)["Role"]
    print(f"Role {GATEWAY_ROLE_NAME} already exists")

In [ ]:
workload_directory = f"arn:aws:bedrock-agentcore:{region}:{account}:workload-identity-directory/default"
workload_identity = [workload_directory, f"{workload_directory}/workload-identity/{GATEWAY_NAME}-*"]
token_vault = provider_arn.split("/apikeycredentialprovider/")[0]

api_key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetWorkloadAccessToken", "Resource": workload_identity},
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetResourceApiKey", "Resource": [provider_arn, token_vault, *workload_identity]},
        {"Effect": "Allow", "Action": "secretsmanager:GetSecretValue", "Resource": secret_arn},
    ],
}

context_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": ["lambda:InvokeFunctionUrl", "lambda:InvokeFunction"], "Resource": context_arn}],
}

iam.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="undercover-gateway-api-key", PolicyDocument=json.dumps(api_key_policy))
iam.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="undercover-gateway-context", PolicyDocument=json.dumps(context_policy))
print("The role can fetch the app's API key and invoke the context layer")

### The Gateway

As on Day 2: it speaks MCP, and callers sign their requests with their AWS credentials.

In [ ]:
def wait_for(get_resource, label):
    """Wait while an AgentCore resource is being created or updated, then print its status."""
    while True:
        resource = get_resource()
        if resource["status"] not in ("CREATING", "UPDATING", "SYNCHRONIZING"):
            break
        time.sleep(3)
    print(f"{label}: {resource['status']}")
    for reason in resource.get("statusReasons", []):
        print(f"  {reason}")
    return resource


def find_gateways():
    """The gateways named undercover-gateway."""
    pages = agentcore.get_paginator("list_gateways").paginate()
    return [g for page in pages for g in page["items"] if g["name"].startswith(GATEWAY_NAME)]

In [ ]:
def create_gateway():
    """Create the Gateway, waiting for its new role to reach AgentCore."""
    for attempt in range(10):
        try:
            return agentcore.create_gateway(
                name=GATEWAY_NAME,
                description="UnderCover's systems as tools for the claim agent",
                roleArn=gateway_role["Arn"],
                protocolType="MCP",
                authorizerType="AWS_IAM",
                exceptionLevel="DEBUG",
                tags=PROJECT_TAG,
            )
        except agentcore.exceptions.ValidationException as error:
            if "role" not in str(error).lower():
                raise
            print("Waiting for the new role to reach AgentCore")
            time.sleep(3)
    raise RuntimeError("AgentCore could not use the role")


existing = find_gateways()
gateway_id = existing[0]["gatewayId"] if existing else create_gateway()["gatewayId"]

gateway = wait_for(lambda: agentcore.get_gateway(gatewayIdentifier=gateway_id), f"Gateway {gateway_id}")
gateway_arn = gateway["gatewayArn"]
gateway_url = gateway["gatewayUrl"]
print(gateway_url)

### The two targets

The app is an OpenAPI target, as on Day 2, with the API key in the `X-API-Key` header. The context layer is an MCP server target: the Gateway asks it for its tools, and signs every call to it with the Gateway's role, for the `lambda` service. A new role or permission can take up to a minute to reach AgentCore, so `put_target` tries again while the Gateway can't use its role yet.

In [ ]:
def find_targets(gateway_id):
    """The Gateway's targets."""
    pages = agentcore.get_paginator("list_gateway_targets").paginate(gatewayIdentifier=gateway_id)
    return [t for page in pages for t in page["items"]]


def put_target(settings):
    """Create a target, or update it in place, and try again while the Gateway's role can't be used yet."""
    for attempt in range(20):
        existing = [t for t in find_targets(gateway_id) if t["name"] == settings["name"]]
        if existing:
            target_id = existing[0]["targetId"]
            agentcore.update_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id, **settings)
        else:
            target_id = agentcore.create_gateway_target(gatewayIdentifier=gateway_id, **settings)["targetId"]
        target = wait_for(lambda: agentcore.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id), f"Target {settings['name']}")
        reasons = " ".join(target.get("statusReasons", []))
        if target["status"] == "READY" or ("role" not in reasons and "Authorization" not in reasons):
            return target
        print("Waiting for the Gateway's role to reach AgentCore")
        time.sleep(10)
    raise RuntimeError("AgentCore could not use the role")

In [ ]:
app_target = put_target(
    {
        "name": APP_TARGET,
        "description": "The UnderCover app's routes for the claim agent",
        "targetConfiguration": {"mcp": {"openApiSchema": {"inlinePayload": json.dumps(agent_api)}}},
        "credentialProviderConfigurations": [
            {
                "credentialProviderType": "API_KEY",
                "credentialProvider": {
                    "apiKeyCredentialProvider": {
                        "providerArn": provider_arn,
                        "credentialParameterName": "X-API-Key",
                        "credentialLocation": "HEADER",
                    }
                },
            }
        ],
    }
)

In [ ]:
context_target = put_target(
    {
        "name": CONTEXT_TARGET,
        "description": "UnderCover's context layer: definitions, plans, rules, reference data and documents",
        "targetConfiguration": {"mcp": {"mcpServer": {"endpoint": f"{context_url}/mcp"}}},
        "credentialProviderConfigurations": [
            {
                "credentialProviderType": "GATEWAY_IAM_ROLE",
                "credentialProvider": {"iamCredentialProvider": {"service": "lambda", "region": region}},
            }
        ],
    }
)

### The Policy

Day 2's Policy, enforced from the start this time. With a policy engine attached, a call is refused unless a policy permits it, so there's a permit for each target's tools, and the forbid on any payment over $1,000.

In [ ]:
def find_policy_engines():
    """The policy engines named undercover..."""
    pages = agentcore.get_paginator("list_policy_engines").paginate()
    return [e for page in pages for e in page["policyEngines"] if e["name"].startswith("undercover")]


existing = [e for e in find_policy_engines() if e["name"] == ENGINE_NAME]

if existing:
    engine_id = existing[0]["policyEngineId"]
else:
    engine_id = agentcore.create_policy_engine(name=ENGINE_NAME, description="Rules for the UnderCover gateway", tags=PROJECT_TAG)["policyEngineId"]

engine = wait_for(lambda: agentcore.get_policy_engine(policyEngineId=engine_id), f"Policy engine {engine_id}")
engine_arn = engine["policyEngineArn"]

policy_engine_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetPolicyEngine", "Resource": engine_arn},
        {
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:AuthorizeAction", "bedrock-agentcore:PartiallyAuthorizeActions"],
            "Resource": [engine_arn, gateway_arn],
        },
    ],
}

iam.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="undercover-gateway-policy-engine", PolicyDocument=json.dumps(policy_engine_policy))
print("The role can use the policy engine")

In [ ]:
for attempt in range(20):
    try:
        agentcore.update_gateway(
            gatewayIdentifier=gateway_id,
            name=GATEWAY_NAME,
            description=gateway["description"],
            roleArn=gateway_role["Arn"],
            protocolType="MCP",
            authorizerType="AWS_IAM",
            exceptionLevel="DEBUG",
            policyEngineConfiguration={"arn": engine_arn, "mode": "ENFORCE"},
        )
        break
    except agentcore.exceptions.ValidationException as error:
        if "GetPolicyEngine" not in str(error):
            raise
        print("Waiting for the role's new permission to reach AgentCore")
        time.sleep(5)

updated = wait_for(lambda: agentcore.get_gateway(gatewayIdentifier=gateway_id), f"Gateway {gateway_id}")
print(f"Policy engine mode: {updated['policyEngineConfiguration']['mode']}")

In [ ]:
def permit(target):
    """A Cedar policy that permits every tool on one target."""
    return f"""permit(
  principal is AgentCore::IamEntity,
  action in AgentCore::Action::"{target}",
  resource == AgentCore::Gateway::"{gateway_arn}"
);"""


policies = {
    "undercover_allow_app_tools": permit(APP_TARGET),
    "undercover_allow_context_tools": permit(CONTEXT_TARGET),
    "undercover_payment_limit": f"""forbid(
  principal is AgentCore::IamEntity,
  action == AgentCore::Action::"{APP_TARGET}___pay_claim",
  resource == AgentCore::Gateway::"{gateway_arn}"
)
when {{
  context.input.amount.greaterThan(decimal("1000.00"))
}};""",
}


def find_policies(engine_id):
    """The policies in a policy engine."""
    pages = agentcore.get_paginator("list_policies").paginate(policyEngineId=engine_id)
    return [p for page in pages for p in page["policies"]]


existing = {p["name"]: p for p in find_policies(engine_id)}

for name, statement in policies.items():
    if name in existing:
        policy_id = existing[name]["policyId"]
    else:
        policy_id = agentcore.create_policy(policyEngineId=engine_id, name=name, definition={"cedar": {"statement": statement}})["policyId"]
    wait_for(lambda: agentcore.get_policy(policyEngineId=engine_id, policyId=policy_id), f"Policy {name}")

### The tools the agent sees

The same MCP client as Day 2. The agent now sees one list of tools from two targets, each named after its target.

In [ ]:
client = MultiServerMCPClient(
    {"undercover": {"transport": "streamable_http", "url": gateway_url, "auth": SigV4Auth(session)}}
)

tools = {tool.name: tool for tool in await client.get_tools()}

for tool in tools.values():
    print(f"{tool.name:<44}{tool.description}")

`call_tool` calls one of them and prints what comes back. An error from the app or the context layer comes back as the tool's result. The Gateway's own refusal is an MCP error.

In [ ]:
async def call_tool(name, **arguments):
    """Call one tool through the Gateway and print what comes back."""
    try:
        result = await tools[name].ainvoke(arguments)
        print(result[0]["text"][:800])
    except McpError as error:
        print(f"Refused by the Gateway: {error}")

A new permission can take up to a minute to reach AgentCore, so the first calls may fail with "Failed to obtain execution role credentials". This cell tries again until they work.

In [ ]:
for attempt in range(20):
    result = await tools[f"{CONTEXT_TARGET}___lookup_term"].ainvoke({"term": "unattended"})
    if "execution role" not in result[0]["text"]:
        break
    print("Waiting for the Gateway's role to reach AgentCore")
    await asyncio.sleep(5)

print(result[0]["text"])

In [ ]:
await call_tool(f"{CONTEXT_TARGET}___read_section", document="claims_manual", number=14)

In [ ]:
await call_tool(f"{CONTEXT_TARGET}___search_reference_data", query="Lisbon strike")

### Which layer says no

The Gateway's Policy guards what the agent does, and the app's check against claims manual section 14 guards what it writes. Both refuse before anything changes. A payment of $4,800 never reaches the app:

In [ ]:
await call_tool(f"{APP_TARGET}___pay_claim", claim_id="CLM-1005", amount=4800.0)

And a decision record the manual wouldn't accept, here a referral that pays something, reaches the app and is refused there by its own check, whatever the claim:

In [ ]:
await call_tool(
    f"{APP_TARGET}___record_decision",
    claim_id="CLM-1005",
    decision="refer",
    payout=243.0,
    section="Claims manual 9",
    reason="A second claim within the year",
    decided_by="Testing from the notebook",
)

## Part 4: The Memory

### Creating it

AgentCore Memory keeps the agent's own notes. After each claim, the agent saves two events: the claim's summary, benefit, date, place and items, and its decision. Each event belongs to an actor, the customer, and a session, the claim. Two strategies turn the events into memories in the background: Semantic keeps facts about each customer, and Summary keeps a summary of each claim, each under a namespace of its own.

Events are kept for 7 days, the shortest AgentCore allows. Creating a Memory takes about two and a half minutes.

In [ ]:
def find_memories():
    """The memories named undercover_memory, leaving out any being deleted."""
    pages = agentcore.get_paginator("list_memories").paginate()
    return [m for page in pages for m in page["memories"] if m["id"].startswith(MEMORY_NAME) and m["status"] != "DELETING"]


existing = find_memories()

if existing:
    memory_id = existing[0]["id"]
else:
    memory_id = agentcore.create_memory(
        name=MEMORY_NAME,
        description="The UnderCover Agent's memory of each customer",
        eventExpiryDuration=7,
        memoryStrategies=[
            {"semanticMemoryStrategy": {"name": "customer_facts", "namespaceTemplates": ["/customers/{actorId}/facts/"]}},
            {"summaryMemoryStrategy": {"name": "claim_summaries", "namespaceTemplates": ["/customers/{actorId}/claims/{sessionId}/"]}},
        ],
        tags=PROJECT_TAG,
    )["memory"]["id"]

while True:
    memory = agentcore.get_memory(memoryId=memory_id)["memory"]
    if memory["status"] != "CREATING":
        break
    time.sleep(10)

print(f"Memory {memory_id}: {memory['status']}")
for strategy in memory["strategies"]:
    print(f"  {strategy['type']:<15}{strategy['namespaces'][0]}")

### Pointing the agent at the Gateway and the Memory

The agent finds the Gateway and the Memory in `.env`. This cell adds the two lines, and Part 6 takes them out again. Your IAM user is the agent's identity on AWS for now: it signs the agent's calls to the Gateway and its calls to the Memory.

In [ ]:
set_key(ENV_FILE, "UNDERCOVER_GATEWAY_URL", gateway_url, quote_mode="never")
set_key(ENV_FILE, "UNDERCOVER_MEMORY_ID", memory_id, quote_mode="never")
print("Added UNDERCOVER_GATEWAY_URL and UNDERCOVER_MEMORY_ID to .env")

## Part 5: A returning customer

Oliver Grant (CUS-3005) has two claims in the queue. In July a giraffe ate his passport in Nairobi, and in August a coconut fell on him in Sri Lanka. His second claim opens "Yes, it is me again".

### A fresh queue

We reseed, so both claims are waiting. Seeding now loads the plans and the two documents from the context layer's files.

In [ ]:
!cd .. && npm run seed

### The first claim

The agent runs on your laptop. Its tools and the payment go through the Gateway, its own records of its work go to the app on Lambda with the API key, and it saves the claim to the Memory once it has decided. Everything it records lands in the same Supabase database the deployed dashboard reads.

In [ ]:
def work(claim_id):
    """Work one claim with the agent on your laptop, against the app on Lambda."""
    environment = os.environ | {"UNDERCOVER_API": app_url}
    result = subprocess.run(["uv", "run", "-m", "agent", claim_id], cwd="..", env=environment, capture_output=True, text=True)
    print(result.stdout + result.stderr)


work("CLM-1005")

Open the claim's page on the deployed dashboard and sign in as Dana Whitfield. Each tool call names the Gateway tool it went through, each lookup says where it came from, and an approval's decision record shows its calculation.

In [ ]:
print(f"{app_url}/claims/CLM-1005")

### What the Memory made of it

AgentCore turns the two events into memories in the background, usually within two minutes. This cell checks every 15 seconds for up to 5 minutes, then shows the facts it kept about Oliver and its summary of the claim. If nothing arrives, look at the agent's output above: it saves nothing when it hasn't recorded a decision.

In [ ]:
facts_namespace = "/customers/CUS-3005/facts/"
summary_namespace = "/customers/CUS-3005/claims/CLM-1005/"


def records_in(namespace):
    """The memory records in one namespace."""
    return memory_records.list_memory_records(memoryId=memory_id, namespace=namespace, maxResults=20)["memoryRecordSummaries"]


for attempt in range(20):
    if records_in(facts_namespace) and records_in(summary_namespace):
        break
    print("Waiting for AgentCore to extract the memories")
    time.sleep(15)

print("Facts about the customer:")
for record in records_in(facts_namespace):
    print(f"- {record['content']['text']}")

print("\nA summary of the claim:")
for record in records_in(summary_namespace):
    print(record["content"]["text"])

### The second claim

The agent decides when to search its memory, with its `search_memory` tool.

In [ ]:
work("CLM-1006")
print(f"{app_url}/claims/CLM-1006")

On CLM-1006's page, look for the memory search and what it found. The agent decides for itself whether to look, so on some runs it doesn't, and the page shows no memory step. The next cell shows what it would have found.

### What the agent can find

The agent's search covers everything under its customer's namespace, the facts and every claim's summary, and nothing else. The same search under another customer finds nothing.

In [ ]:
def search_memory(customer_id, query):
    """Search one customer's memories, as the agent's search_memory tool does."""
    response = memory_records.retrieve_memory_records(
        memoryId=memory_id,
        namespacePath=f"/customers/{customer_id}",
        searchCriteria={"searchQuery": query, "topK": 5},
    )
    return [record["content"]["text"] for record in response["memoryRecordSummaries"]]


for customer_id in ["CUS-3005", "CUS-3001"]:
    found = search_memory(customer_id, "earlier claims")
    print(f"{customer_id}: {len(found)} memories")
    for text in found:
        print(f"  {text[:200]}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Three sources of data</h2>
            <span style="color:#118AB2;">The context layer answered the agent's lookups through the Gateway, signed with the Gateway's role, and the agent can only read it. The Memory took the agent's notes directly, signed with your IAM user, and keeps each customer apart. The database on Supabase holds the records, which the agent reaches through the app, and the checkpoints of its work. Each has a different owner: data governance, the agent and the app.
            </span>
        </td>
    </tr>
</table>

## Part 6: Teardown

The app and the context layer stay on Lambda for Day 4, and cost nothing while nobody calls them. We delete the AgentCore pieces, which Day 4 creates again, and take the two lines out of `.env`, so the agent goes back to calling the app directly.

In [ ]:
def find_roles():
    """The IAM roles named undercover..."""
    pages = iam.get_paginator("list_roles").paginate()
    return [r["RoleName"] for page in pages for r in page["Roles"] if r["RoleName"].startswith("undercover")]


def show_resources():
    """Print what today's lab created that still exists."""
    lambda_pages = lambda_client.get_paginator("list_functions").paginate()
    found = {
        "Lambda functions": [f["FunctionName"] for page in lambda_pages for f in page["Functions"] if f["FunctionName"].startswith("undercover")],
        "Gateways": [g["name"] for g in find_gateways()],
        "Policy engines": [e["name"] for e in find_policy_engines()],
        "API key providers": find_key_providers(),
        "Memories": [m["id"] for m in find_memories()],
        "IAM roles": find_roles(),
    }
    for label, names in found.items():
        print(f"{label + ':':<20}{', '.join(names) or 'none'}")


show_resources()

Targets go before their gateway and policies before their engine. AgentCore deletes the secret that held the API key a few seconds after the provider, and a Memory's events and records go with it.

In [ ]:
for summary in find_gateways():
    for target_summary in find_targets(summary["gatewayId"]):
        agentcore.delete_gateway_target(gatewayIdentifier=summary["gatewayId"], targetId=target_summary["targetId"])
        print(f"Deleted target {target_summary['name']}")
    while find_targets(summary["gatewayId"]):
        time.sleep(3)
    agentcore.delete_gateway(gatewayIdentifier=summary["gatewayId"])
    print(f"Deleted gateway {summary['name']}")

for summary in find_policy_engines():
    for policy_summary in find_policies(summary["policyEngineId"]):
        agentcore.delete_policy(policyEngineId=summary["policyEngineId"], policyId=policy_summary["policyId"])
        print(f"Deleted policy {policy_summary['name']}")
    while find_policies(summary["policyEngineId"]):
        time.sleep(3)
    agentcore.delete_policy_engine(policyEngineId=summary["policyEngineId"])
    print(f"Deleted policy engine {summary['name']}")

for name in find_key_providers():
    agentcore.delete_api_key_credential_provider(name=name)
    print(f"Deleted credential provider {name}")

for memory_summary in find_memories():
    agentcore.delete_memory(memoryId=memory_summary["id"])
    print(f"Deleted memory {memory_summary['id']}")

if GATEWAY_ROLE_NAME in find_roles():
    for policy_name in iam.list_role_policies(RoleName=GATEWAY_ROLE_NAME)["PolicyNames"]:
        iam.delete_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName=policy_name)
    iam.delete_role(RoleName=GATEWAY_ROLE_NAME)
    print(f"Deleted role {GATEWAY_ROLE_NAME}")

unset_key(ENV_FILE, "UNDERCOVER_GATEWAY_URL")
unset_key(ENV_FILE, "UNDERCOVER_MEMORY_ID")
print("Removed UNDERCOVER_GATEWAY_URL and UNDERCOVER_MEMORY_ID from .env")

If you're stopping the course here, set `DELETE_THE_FUNCTIONS` to `True` and run the cell to delete the app and the context layer from Lambda as well, with their roles and log groups. Deleting a function also deletes its address.

In [ ]:
DELETE_THE_FUNCTIONS = False

if DELETE_THE_FUNCTIONS:
    for function, role_name, log_group in [
        (APP_FUNCTION, APP_ROLE_NAME, APP_LOG_GROUP),
        (CONTEXT_FUNCTION, CONTEXT_ROLE_NAME, CONTEXT_LOG_GROUP),
    ]:
        lambda_client.delete_function(FunctionName=function)
        print(f"Deleted function {function}")
        for policy_name in iam.list_role_policies(RoleName=role_name)["PolicyNames"]:
            iam.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
        iam.delete_role(RoleName=role_name)
        print(f"Deleted role {role_name}")
        logs.delete_log_group(logGroupName=log_group)
        print(f"Deleted log group {log_group}")

In [ ]:
time.sleep(10)
show_resources()

Over the next day or two, keep an eye on your spend with `uv run monitor.py`. For the official figures, review Billing and Cost Management in your AWS console.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: add AgentCore's Episodic strategy to the Memory with its own namespace for each customer, run Oliver Grant's two claims again, and compare what the Episodic strategy keeps with the Semantic facts and the Summary.<br/><br/>I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>